In [3]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
)

# Load dataset
df = pd.read_csv("dataset.csv")
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
# Drop ID columns and leakage failure modes
drop_cols = ["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Feature engineering to exceed the >= 12 features requirement
df["Temp_diff"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["Power_W"] = df["Torque [Nm]"] * (df["Rotational speed [rpm]"] * (2 * np.pi / 60))
df["Overstrain"] = df["Tool wear [min]"] * df["Torque [Nm]"]
df["Temp_ratio"] = df["Process temperature [K]"] / df["Air temperature [K]"]
df["Speed_Torque_ratio"] = df["Rotational speed [rpm]"] / (df["Torque [Nm]"] + 1e-5)
df["Wear_Speed_prod"] = df["Tool wear [min]"] * df["Rotational speed [rpm]"]

# One-hot encode the 'Type' column
df = pd.get_dummies(df, columns=["Type"], drop_first=False)

target_col = "Machine failure"
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"Total instances: {len(df)}")
print(f"Total predictive features: {X.shape[1]}")

Total instances: 10000
Total predictive features: 14


In [5]:
# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Export test_data.csv for your Streamlit app
test_df = X_test.copy()
test_df[target_col] = y_test
test_df.to_csv("test_data.csv", index=False)

print("test_data.csv generated successfully!")

test_data.csv generated successfully!


In [6]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create model directory and persist scaler
os.makedirs("model", exist_ok=True)
joblib.dump(scaler, "model/preprocessor.joblib")

# 5 Mandatory Models
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=6, class_weight="balanced"),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(probability=True, random_state=42, class_weight="balanced"),
}

results = []
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    
    # Save model artifact
    file_name = name.lower().replace(" ", "_")
    joblib.dump(model, f"model/{file_name}.joblib")
    
    # Evaluation
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    results.append({
        "ML Model Name": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_proba), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4),
    })

results_df = pd.DataFrame(results)
print("\n=== FINAL EVALUATION METRICS TABLE ===")
display(results_df)

Training Logistic Regression...
Training Decision Tree...
Training KNN...
Training Naive Bayes...
Training Support Vector Machine...

=== FINAL EVALUATION METRICS TABLE ===


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.8550,0.9414,0.1754,0.8824,0.2927,0.3545
1,Decision Tree,0.9550,0.8576,0.4179,0.8235,0.5545,0.5677
2,KNN,0.9735,0.8678,0.8261,0.2794,0.4176,0.4714
3,Naive Bayes,0.9510,0.9201,0.3214,0.3971,0.3553,0.3321
4,Support Vector Machine,0.9200,0.9697,0.2870,0.9118,0.4366,0.4858


In [7]:
%%writefile requirements.txt
streamlit>=1.30.0
scikit-learn>=1.3.0
pandas>=2.0.0
numpy>=1.24.0
matplotlib>=3.7.0
seaborn>=0.12.0
joblib>=1.3.0

Writing requirements.txt
